# 02 — STORAGE LAYER & DATA QUALITY
# Zona bronze → silver · penggabungan dua jalur · gerbang kualitas

Notebook ini mengubah keluaran mentah notebook 01 menjadi satu tabel Parquet bersih yang siap dipakai seluruh notebook berikutnya.

**Pekerjaan intinya adalah menyusun kembali nomor segmen dan subtipe**, karena tidak satu sumber pun menyediakannya secara lengkap:

- `Segment_s` dari jalur B kosong pada sebagian besar rekaman lama — pada partisi 1995 hanya 11% yang terisi.
- Defline FASTA jalur A kadang menyebut `segment 4`, kadang hanya menyebut nama gen (`HA`, `neuraminidase`), kadang tidak keduanya.
- `Serotype_s` terisi jauh lebih baik, tetapi tetap tidak seluruhnya.

Karena itu setiap rekaman melewati **rantai sumber berjenjang**, dan kolom `segmen_sumber` serta `subtipe_sumber` mencatat dari mana nilainya berasal. Tabel asal-usul itulah isi Bab 3.4 — bukan pernyataan normatif, melainkan angka.


## Bootstrap dan sesi Spark

In [1]:
import sys
sys.path.insert(0, r"D:\BDA\nb")
from bda_common import *

from pyspark.sql import functions as F, types as T

info_mesin()
spark = spark_session("02-storage-quality")
print()
print("  Sumber  :", jalur("lake/fasta"))
print("  Tujuan  :", jalur("stage/sequences"))

CPU logis      : 24
RAM total      : 50.5 GB   bebas 44.1 GB
Disk D: bebas  : 201.4 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20260922T021119Z


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 02:11:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 02:11:22 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark 3.5.9 | master yarn | driver 8g | paralelisme 8
Spark UI: http://jupyter:4040

  Sumber  : hdfs://namenode:8020/bda/lake/fasta
  Tujuan  : hdfs://namenode:8020/bda/stage/sequences


---
## Menaikkan zona bronze ke HDFS

Notebook 01 menulis hasil ingestion sebagai berkas biasa di `D:\BDA\lake`, yang dari dalam klaster terlihat sebagai `/workspace/lake`. Berkas itu belum berada di HDFS.

Sel berikut memindahkannya. Penyalinan dilakukan lewat API FileSystem Hadoop di JVM Spark, bukan perkakas baris perintah `hdfs` — image jupyter tidak memuatnya, sementara pustaka Hadoop sudah dibawa Spark. Berkas yang sudah ada di HDFS dilewati, jadi sel ini aman dijalankan berulang kali setiap selesai menarik partisi baru.

Saat berjalan di mode lokal Windows, sel ini otomatis dilewati karena zona lake memang sudah berada di tempat yang dibaca Spark.

In [2]:
with Tahap("naikkan zona bronze ke HDFS", "STORAGE"):
    if KLASTER:
        unggah_ke_hdfs(spark, LAKE_FASTA, jalur("lake/fasta"), "*.fasta.gz")
        unggah_ke_hdfs(spark, LAKE_META, jalur("lake/meta_csv"), "*.csv")
    else:
        print("  mode lokal -- zona lake sudah dibaca langsung, tidak perlu diunggah")


--------------------------------------------------------------------
[>] STORAGE | naikkan zona bronze ke HDFS
  fasta: 0 berkas baru diunggah, 152 sudah ada -> hdfs://namenode:8020/bda/lake/fasta
  meta_csv: 0 berkas baru diunggah, 49 sudah ada -> hdfs://namenode:8020/bda/lake/meta_csv
[<] OK | 0.5 detik | RAM bebas 42.2 GB


---
## LAPISAN 3A — Membaca FASTA tanpa memuatnya ke memori

FASTA adalah format multi-baris: satu rekaman terdiri dari satu defline lalu sejumlah baris sekuens. Trik yang membuatnya bisa dibaca Spark secara paralel adalah mengganti pemisah baris menjadi tanda `>`, sehingga **satu rekaman FASTA menjadi satu baris DataFrame**.

In [3]:
with Tahap("baca FASTA jalur A", "STORAGE"):
    mentah = (spark.read
              .option("lineSep", ">")
              .text(jalur("lake/fasta") + "/*.fasta.gz"))

    fasta = (mentah
             .filter(F.length("value") > 20)
             .withColumn("_p", F.split(F.col("value"), "\n", 2))
             .withColumn("defline", F.trim(F.col("_p")[0]))
             .withColumn("seq", F.upper(F.regexp_replace(F.col("_p")[1], r"\s", "")))
             .drop("_p", "value")
             .withColumn("accver", F.regexp_extract("defline", r"^(\S+)", 1))
             .withColumn("accession", F.split(F.col("accver"), r"\.")[0])
             .filter(F.col("accession") != "")
             .withColumn("panjang", F.length("seq"))
             .withColumn("n_acgt", F.length(F.regexp_replace("seq", "[^ACGT]", "")))
             .withColumn("pct_ambigu",
                         F.round(100 * (F.col("panjang") - F.col("n_acgt"))
                                 / F.greatest(F.col("panjang"), F.lit(1)), 3))
             .withColumn("gc_pct",
                         F.round(100 * F.length(F.regexp_replace("seq", "[^GC]", ""))
                                 / F.greatest(F.col("panjang"), F.lit(1)), 2)))

    fasta.cache()
    n_fasta = fasta.count()
    print(f"  Rekaman FASTA terbaca : {n_fasta:,}")
    fasta.select("accession", "panjang", "pct_ambigu", "gc_pct").show(3, truncate=False)


--------------------------------------------------------------------
[>] STORAGE | baca FASTA jalur A


  Rekaman FASTA terbaca : 1,627,536
+---------+-------+----------+------+
|accession|panjang|pct_ambigu|gc_pct|
+---------+-------+----------+------+
|KC669390 |1565   |0.0       |47.41 |
|KC669389 |1698   |0.0       |43.46 |
|KC669388 |865    |0.0       |44.51 |
+---------+-------+----------+------+
only showing top 3 rows

[<] OK | 53.3 detik | RAM bebas 35.4 GB


## LAPISAN 3B — Membaca metadata terstruktur

Kolom `Definition_s` berisi judul GenBank yang mengandung koma dan tanda kutip, jadi CSV harus dibaca dengan penanganan kutip yang benar. Tanpa `multiLine` dan `escape`, satu judul bertanda kutip bisa menggeser seluruh kolom di baris-baris berikutnya — kegagalan senyap yang klasik.

In [4]:
with Tahap("baca metadata jalur B", "STORAGE"):
    meta = (spark.read
            .option("header", True)
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .option("mode", "PERMISSIVE")
            .csv(jalur("lake/meta_csv") + "/*.csv"))

    meta = (meta
            .withColumn("accession", F.split(F.col("AccVer_s"), r"\.")[0])
            .filter(F.col("accession").isNotNull() & (F.col("accession") != ""))
            .withColumnRenamed("Serotype_s", "vv_subtipe")
            .withColumnRenamed("Segment_s", "vv_segmen")
            .withColumnRenamed("Host_s", "inang")
            .withColumnRenamed("CountryFull_s", "lokasi_penuh")
            .withColumnRenamed("Region_s", "wilayah")
            .withColumnRenamed("CollectionDate_s", "tgl_koleksi")
            .withColumnRenamed("Isolate_s", "isolat")
            .withColumnRenamed("CreateDate_dt", "tgl_rilis")
            .withColumnRenamed("Completeness_s", "kelengkapan")
            .select("accession", "vv_subtipe", "vv_segmen", "inang", "lokasi_penuh",
                    "wilayah", "tgl_koleksi", "tgl_rilis", "kelengkapan", "isolat"))

    # satu aksesi harus muncul sekali; ambil baris terlengkap bila ada duplikat
    from pyspark.sql import Window
    w = Window.partitionBy("accession").orderBy(
        F.desc(F.length(F.coalesce(F.col("vv_subtipe"), F.lit("")))),
        F.desc(F.length(F.coalesce(F.col("vv_segmen"), F.lit("")))))
    meta = (meta.withColumn("_r", F.row_number().over(w))
                .filter(F.col("_r") == 1).drop("_r"))

    meta.cache()
    n_meta = meta.count()
    print(f"  Baris metadata unik : {n_meta:,}")


--------------------------------------------------------------------
[>] STORAGE | baca metadata jalur B


[Stage 10:=====================================================>(255 + 1) / 256]

  Baris metadata unik : 1,641,553
[<] OK | 6.0 detik | RAM bebas 33.8 GB


## LAPISAN 3C — Penggabungan dan rantai sumber berjenjang

Kedua jalur digabung lewat nomor aksesi tanpa versi. Lalu setiap kolom kunci diisi dari sumber pertama yang tersedia, dan asal-usulnya dicatat.

**Segmen** diambil berurutan dari: kolom `Segment_s` → kata `segment N` di defline → nama gen di defline, tetapi **hanya bila tepat satu nama gen cocok**. Defline seperti `HA-NP=fusion gene` menyebut dua gen sekaligus, dan menebak salah satunya akan menyuntikkan label palsu ke data latih. Rekaman semacam itu dibiarkan kosong dengan sengaja.

**Subtipe** diambil dari `Serotype_s` → pola `H#N#` di defline.

In [5]:
GEN_SEGMEN = [
    (1, r"\b(PB2|polymerase\s+basic\s+2)\b"),
    (2, r"\b(PB1|polymerase\s+basic\s+1)\b"),
    (3, r"\b(PA|polymerase\s+acidic)\b"),
    (4, r"\b(HA|haemagglutinin|hemagglutinin)\b"),
    (5, r"\b(NP|nucleoprotein|nucleocapsid)\b"),
    (6, r"\b(NA|neuraminidase)\b"),
    (7, r"\b(M1|M2|MP|matrix)\b"),
    (8, r"\b(NS1|NS2|non-?structural|nonstructural)\b"),
]

peta_seg = F.create_map([F.lit(x) for kv in PETA_SEGMEN.items() for x in
                         (kv[0], int(kv[1]))])

with Tahap("gabung dua jalur + rantai sumber", "STORAGE"):
    df = fasta.join(meta, on="accession", how="left")

    df = df.withColumn("seg_vv", F.element_at(
        peta_seg, F.upper(F.trim(F.coalesce(F.col("vv_segmen"), F.lit(""))))))
    df = df.withColumn(
        "seg_defline",
        F.regexp_extract(F.col("defline"), r"(?i)\bsegment\s+([1-8])\b", 1)
         .cast("int"))
    df = df.withColumn("seg_defline",
                       F.when(F.col("seg_defline") == 0, None).otherwise(F.col("seg_defline")))

    for n, pola in GEN_SEGMEN:
        df = df.withColumn(f"_g{n}",
                           F.col("defline").rlike("(?i)" + pola).cast("int"))
    df = df.withColumn("_n_gen", sum(F.col(f"_g{n}") for n, _ in GEN_SEGMEN))
    ekspr_gen = F.lit(None).cast("int")
    for n, _ in GEN_SEGMEN:
        ekspr_gen = F.when(F.col(f"_g{n}") == 1, F.lit(n)).otherwise(ekspr_gen)
    df = df.withColumn("seg_gen",
                       F.when(F.col("_n_gen") == 1, ekspr_gen).otherwise(F.lit(None)))
    df = df.drop(*[f"_g{n}" for n, _ in GEN_SEGMEN])

    df = (df
          .withColumn("segmen", F.coalesce("seg_vv", "seg_defline", "seg_gen"))
          .withColumn("segmen_sumber",
                      F.when(F.col("seg_vv").isNotNull(), "metadata")
                       .when(F.col("seg_defline").isNotNull(), "defline_segment")
                       .when(F.col("seg_gen").isNotNull(), "defline_gen")
                       .otherwise("tidak_diketahui")))

    df = (df
          .withColumn("sub_vv", F.upper(F.regexp_extract(
              F.coalesce(F.col("vv_subtipe"), F.lit("")), r"(?i)\b(H\d{1,2}N\d{1,2})\b", 1)))
          .withColumn("sub_defline", F.upper(F.regexp_extract(
              F.col("defline"), r"(?i)\b(H\d{1,2}N\d{1,2})\b", 1))))
    for c in ("sub_vv", "sub_defline"):
        df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
    df = (df
          .withColumn("subtipe", F.coalesce("sub_vv", "sub_defline"))
          .withColumn("subtipe_sumber",
                      F.when(F.col("sub_vv").isNotNull(), "metadata")
                       .when(F.col("sub_defline").isNotNull(), "defline")
                       .otherwise("tidak_diketahui")))

    df = (df
          .withColumn("tahun_koleksi",
                      F.regexp_extract(F.coalesce("tgl_koleksi", F.lit("")), r"(\d{4})", 1)
                       .cast("int"))
          .withColumn("presisi_tanggal",
                      F.when(F.col("tgl_koleksi").rlike(r"^\d{4}-\d{2}-\d{2}"), "hari")
                       .when(F.col("tgl_koleksi").rlike(r"^\d{4}-\d{2}$"), "bulan")
                       .when(F.col("tgl_koleksi").rlike(r"^\d{4}$"), "tahun")
                       .otherwise("tidak_ada"))
          .withColumn("tahun_rilis",
                      F.regexp_extract(F.coalesce("tgl_rilis", F.lit("")), r"(\d{4})", 1)
                       .cast("int"))
          .withColumn("negara", F.trim(F.split(F.coalesce("lokasi_penuh", F.lit("")), ":")[0])))
    df = df.withColumn("negara", F.when(F.col("negara") == "", None).otherwise(F.col("negara")))

    df.cache()
    n_gabung = df.count()
    print(f"  Baris setelah penggabungan : {n_gabung:,}")
    print(f"  Cocok dengan metadata      : {df.filter(F.col('vv_subtipe').isNotNull() | F.col('vv_segmen').isNotNull()).count():,}")


--------------------------------------------------------------------
[>] STORAGE | gabung dua jalur + rantai sumber


26/09/22 02:12:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  Baris setelah penggabungan : 1,627,536
  Cocok dengan metadata      : 1,590,956
[<] OK | 16.8 detik | RAM bebas 30.7 GB


In [6]:
with Tahap("tabel asal-usul label", "DATA QUALITY"):
    print("  Asal nomor segmen:")
    asal_seg = (df.groupBy("segmen_sumber").count()
                  .withColumn("pct", F.round(100 * F.col("count") / n_gabung, 2))
                  .orderBy(F.desc("count")))
    asal_seg.show(truncate=False)

    print("  Asal subtipe:")
    asal_sub = (df.groupBy("subtipe_sumber").count()
                  .withColumn("pct", F.round(100 * F.col("count") / n_gabung, 2))
                  .orderBy(F.desc("count")))
    asal_sub.show(truncate=False)

    n_seg_tanpa_meta = df.filter(
        (F.col("segmen_sumber") != "metadata") & F.col("segmen").isNotNull()).count()
    n_sub_tanpa_meta = df.filter(F.col("subtipe_sumber") == "defline").count()
    print(f"  Segmen yang diselamatkan defline : {n_seg_tanpa_meta:,}")
    print(f"  Subtipe yang diselamatkan defline: {n_sub_tanpa_meta:,}")
    print("\n  Tanpa rantai berjenjang ini, seluruh rekaman tersebut akan terbuang.")


--------------------------------------------------------------------
[>] DATA QUALITY | tabel asal-usul label
  Asal nomor segmen:
+---------------+-------+-----+
|segmen_sumber  |count  |pct  |
+---------------+-------+-----+
|metadata       |1573329|96.67|
|tidak_diketahui|30794  |1.89 |
|defline_gen    |22830  |1.4  |
|defline_segment|583    |0.04 |
+---------------+-------+-----+

  Asal subtipe:
+---------------+-------+-----+
|subtipe_sumber |count  |pct  |
+---------------+-------+-----+
|metadata       |1540473|94.65|
|tidak_diketahui|78661  |4.83 |
|defline        |8402   |0.52 |
+---------------+-------+-----+

  Segmen yang diselamatkan defline : 23,413
  Subtipe yang diselamatkan defline: 8,402

  Tanpa rantai berjenjang ini, seluruh rekaman tersebut akan terbuang.
[<] OK | 2.4 detik | RAM bebas 30.5 GB


---
## GERBANG KUALITAS DATA
### Setiap aturan dihitung, bukan sekadar dinyatakan

Lima aturan di bawah dijalankan berurutan, dan jumlah baris yang gugur pada setiap aturan dicatat. Tabel hasilnya masuk langsung ke Bab 3.4.

Alasan tiap ambang:

| Aturan | Ambang | Alasan |
|---|---|---|
| Panjang minimum | 500 bp | Vektor *k*-mer dari sekuens sangat pendek terlalu jarang untuk dipercaya |
| Panjang maksimum | 2.600 bp | Segmen influenza terpanjang 2.341 bp; di atas ini menandakan entri gabungan atau kontaminasi vektor |
| Huruf ambigu | ≤ 5% | N dan kode IUPAC lain merusak hitungan *k*-mer |
| Duplikat aksesi | — | Satu aksesi satu baris |
| Duplikat sekuens | — | Sekuens identik yang disetorkan berkali-kali membocorkan data latih ke data uji |

In [7]:
with Tahap("gerbang kualitas data", "DATA QUALITY"):
    aturan, sisa = [], df
    n0 = n_gabung

    def gerbang(nama, kondisi, alasan):
        global sisa
        sebelum = sisa.count() if not aturan else aturan[-1]["sisa"]
        sisa = sisa.filter(kondisi)
        sesudah = sisa.count()
        aturan.append({"aturan": nama, "gugur": sebelum - sesudah,
                       "sisa": sesudah,
                       "pct_gugur": round(100 * (sebelum - sesudah) / max(n0, 1), 3),
                       "alasan": alasan})
        print(f"  {nama:<28} gugur {sebelum-sesudah:>9,}  sisa {sesudah:>9,}")

    gerbang("panjang >= 500 bp", F.col("panjang") >= CFG["min_len"],
            "vektor k-mer terlalu jarang di bawah ini")
    gerbang("panjang <= 2600 bp", F.col("panjang") <= CFG["max_len"],
            "melebihi segmen terpanjang influenza A")
    gerbang("ambigu <= 5%", F.col("pct_ambigu") <= CFG["max_ambigu_pct"],
            "huruf non-ACGT merusak hitungan k-mer")

    # Duplikat nomor aksesi tetap dibuang: satu aksesi harus satu baris.
    sebelum = aturan[-1]["sisa"]
    from pyspark.sql import Window
    w = Window.partitionBy("accession").orderBy(F.desc("panjang"))
    sisa = sisa.withColumn("_r", F.row_number().over(w)).filter(F.col("_r") == 1).drop("_r")
    sesudah = sisa.count()
    aturan.append({"aturan": "duplikat aksesi", "gugur": sebelum - sesudah,
                   "sisa": sesudah, "pct_gugur": round(100*(sebelum-sesudah)/max(n0,1), 3),
                   "alasan": "satu aksesi satu baris"})
    print(f"  {'duplikat aksesi':<28} gugur {sebelum-sesudah:>9,}  sisa {sesudah:>9,}")

    sisa = sisa.withColumn("hash_seq", F.sha2(F.col("seq"), 256))
    w2 = Window.partitionBy("hash_seq").orderBy("accession")
    sisa = (sisa.withColumn("_r", F.row_number().over(w2))
                .withColumn("n_duplikat_seq",
                            F.count("*").over(Window.partitionBy("hash_seq")))
                .withColumn("wakil_unik", F.col("_r") == 1)
                .drop("_r"))
    n_wakil = sisa.filter(F.col("wakil_unik")).count()
    aturan.append({"aturan": "duplikat sekuens", "gugur": 0, "sisa": sesudah,
                   "pct_gugur": 0.0,
                   "alasan": f"ditandai, tidak dibuang -- {n_wakil:,} wakil unik "
                             f"dari {sesudah:,} baris"})
    print(f"  {'duplikat sekuens':<28} gugur {0:>9,}  sisa {sesudah:>9,}"
          f"   (ditandai: {n_wakil:,} wakil unik)")

    import pandas as pd
    df_aturan = pd.DataFrame(aturan)
    print()
    print(f"  Masuk {n0:,} -> lolos {sesudah:,} ({100*sesudah/max(n0,1):.1f}%)")
    print(f"  Di antaranya {n_wakil:,} sekuens unik "
          f"({100*n_wakil/max(sesudah,1):.1f}%), sisanya salinan identik")
    display(df_aturan)


--------------------------------------------------------------------
[>] DATA QUALITY | gerbang kualitas data
  panjang >= 500 bp            gugur    36,388  sisa 1,591,148
  panjang <= 2600 bp           gugur        49  sisa 1,591,099
  ambigu <= 5%                 gugur     4,187  sisa 1,586,912


  duplikat aksesi              gugur         0  sisa 1,586,912


[Stage 99:===================================================>  (244 + 4) / 256]

  duplikat sekuens             gugur         0  sisa 1,586,912   (ditandai: 893,599 wakil unik)

  Masuk 1,627,536 -> lolos 1,586,912 (97.5%)
  Di antaranya 893,599 sekuens unik (56.3%), sisanya salinan identik


,aturan,gugur,sisa,pct_gugur,alasan
0,panjang >= 500 bp,36388,1591148,2.236,vektor k-mer terlalu jarang di bawah ini
1,panjang <= 2600 bp,49,1591099,0.003,melebihi segmen terpanjang influenza A
2,ambigu <= 5%,4187,1586912,0.257,huruf non-ACGT merusak hitungan k-mer
3,duplikat aksesi,0,1586912,0.000,satu aksesi satu baris
4,duplikat sekuens,0,1586912,0.000,"ditandai, tidak dibuang -- 893,599 wakil unik ..."


[<] OK | 6.2 detik | RAM bebas 29.4 GB


## Model data — zona silver

Tabel ditulis sebagai Parquet **dipartisi menurut nomor segmen**. Ini bukan pilihan kosmetik: notebook 05 melatih model terpisah untuk gen eksternal (segmen 4 dan 6) dan gen internal (1, 2, 3, 5, 7, 8), sehingga partisi ini membuat Spark hanya membaca partisi yang diperlukan — *partition pruning* yang bisa Anda tunjukkan di Spark UI sebagai bukti optimasi.

Rekaman yang segmennya tidak diketahui tetap disimpan di partisi `segmen=-1`. Rekaman itu tidak dipakai melatih, tetapi justru menjadi sasaran **imputasi** oleh model klasifikasi segmen di notebook 05.

In [8]:
with Tahap("tulis zona silver", "STORAGE"):
    silver = (sisa
              .withColumn("segmen_part", F.coalesce(F.col("segmen"), F.lit(-1)))
              .select("accession", "accver", "defline", "seq", "panjang", "n_acgt",
                      "pct_ambigu", "gc_pct", "hash_seq", "n_duplikat_seq",
                      "wakil_unik",
                      "segmen", "segmen_sumber", "subtipe", "subtipe_sumber",
                      "inang", "negara", "wilayah", "lokasi_penuh", "isolat",
                      "tgl_koleksi", "tahun_koleksi", "presisi_tanggal",
                      "tahun_rilis", "kelengkapan", "segmen_part"))

    tujuan = jalur("stage/sequences")
    (silver.repartition("segmen_part")
           .write.mode("overwrite")
           .partitionBy("segmen_part")
           .parquet(tujuan))

    print(f"  Ditulis ke {tujuan}")
    if not KLASTER:
        print(f"  Ukuran     : {ukuran(Path(tujuan))}")


--------------------------------------------------------------------
[>] STORAGE | tulis zona silver


[Stage 120:==================================================>      (8 + 1) / 9]

  Ditulis ke hdfs://namenode:8020/bda/stage/sequences
[<] OK | 18.1 detik | RAM bebas 29.3 GB


In [9]:
with Tahap("profil zona silver", "DATA QUALITY"):
    s = spark.read.parquet(jalur("stage/sequences"))
    n = s.count()
    print(f"  Total baris silver: {n:,}\n")

    print("  Sebaran segmen:")
    (s.groupBy("segmen").count().orderBy("segmen")
      .withColumn("pct", F.round(100 * F.col("count") / n, 2)).show(12))

    print("  Subtipe terbanyak:")
    (s.filter(F.col("subtipe").isNotNull()).groupBy("subtipe").count()
      .orderBy(F.desc("count")).show(15, truncate=False))

    print("  Kelengkapan kolom kunci:")
    kolom = ["segmen", "subtipe", "inang", "negara", "tahun_koleksi"]
    isi = s.select([F.round(100 * F.avg(F.col(c).isNotNull().cast("double")), 2).alias(c)
                    for c in kolom]).collect()[0].asDict()
    for k, v in isi.items():
        print(f"    {k:<16} {v:>6.2f}%")

    print("\n  Presisi tanggal koleksi:")
    s.groupBy("presisi_tanggal").count().orderBy(F.desc("count")).show()

    print()
    n_wk = s.filter(F.col("wakil_unik")).count()
    print(f"  Baris seluruhnya      : {n:,}")
    print(f"  Sekuens unik (wakil)  : {n_wk:,}  -> dipakai notebook 04 dan 04")
    print(f"  Salinan identik       : {n - n_wk:,}  -> dipakai notebook 06 dan 06")
    n_neg = s.select("negara").distinct().count()
    print(f"  Negara terwakili      : {n_neg:,}")

    laporan = {
        "run_id": RUN_ID,
        "masuk": int(n_gabung), "lolos": int(n), "wakil_unik": int(n_wk),
        "aturan": df_aturan.to_dict("records"),
        "kelengkapan_pct": isi,
        "asal_segmen": {r["segmen_sumber"]: r["count"] for r in asal_seg.collect()},
        "asal_subtipe": {r["subtipe_sumber"]: r["count"] for r in asal_sub.collect()},
    }
    (STAGE / "laporan_kualitas.json").write_text(
        json.dumps(laporan, indent=2, default=str), encoding="utf-8")
    print(f"\n  Laporan kualitas disimpan: {STAGE / 'laporan_kualitas.json'}")


--------------------------------------------------------------------
[>] DATA QUALITY | profil zona silver
  Total baris silver: 1,586,912

  Sebaran segmen:
+------+------+-----+
|segmen| count|  pct|
+------+------+-----+
|  NULL|  3067| 0.19|
|     1|178568|11.25|
|     2|174269|10.98|
|     3|177554|11.19|
|     4|269754| 17.0|
|     5|183090|11.54|
|     6|217937|13.73|
|     7|197358|12.44|
|     8|185315|11.68|
+------+------+-----+

  Subtipe terbanyak:
+-------+------+
|subtipe|count |
+-------+------+
|H3N2   |568329|
|H1N1   |433463|
|H5N1   |214639|
|H9N2   |66791 |
|H1N2   |47468 |
|H3N8   |32973 |
|H4N6   |18280 |
|H5N2   |15067 |
|H7N9   |10716 |
|H5N8   |8769  |
|H6N2   |8502  |
|H7N3   |8375  |
|H5N6   |7212  |
|H10N7  |7103  |
|H6N1   |6592  |
+-------+------+
only showing top 15 rows

  Kelengkapan kolom kunci:
    segmen            99.81%
    subtipe           96.92%
    inang             96.23%
    negara            98.10%
    tahun_koleksi     97.07%

  Presisi t

In [10]:
display(ringkas_zona())
display(jejak_df())
stop_spark()
print("\nSelesai. Lanjut ke 03_mapreduce_yarn.ipynb")

,zona,isi,ukuran
0,lake/fasta,152,162.2 MB
1,lake/meta_csv,49,367.6 MB
2,stage,1,1.3 KB
3,features,1,735.0 B
4,models,4,2.2 KB
5,graph,0,0.0 B
6,mart,0,0.0 B
7,output,3,237.4 KB


,run_id,lapisan,tahap,status,detik,ram_delta_gb,ram_bebas_gb,waktu
0,run_20260922T021119Z,STORAGE,naikkan zona bronze ke HDFS,OK,0.47,0.00,42.2,2026-09-22T02:11:35.270070+00:00
1,run_20260922T021119Z,STORAGE,baca FASTA jalur A,OK,53.25,6.80,35.4,2026-09-22T02:12:28.532416+00:00
2,run_20260922T021119Z,STORAGE,baca metadata jalur B,OK,5.99,1.57,33.8,2026-09-22T02:12:34.534308+00:00
3,run_20260922T021119Z,STORAGE,gabung dua jalur + rantai sumber,OK,16.77,3.09,30.7,2026-09-22T02:12:51.533434+00:00
4,run_20260922T021119Z,DATA QUALITY,tabel asal-usul label,OK,2.40,0.22,30.5,2026-09-22T02:12:53.940366+00:00
5,run_20260922T021119Z,DATA QUALITY,gerbang kualitas data,OK,6.20,1.08,29.4,2026-09-22T02:13:00.155777+00:00
6,run_20260922T021119Z,STORAGE,tulis zona silver,OK,18.10,0.12,29.3,2026-09-22T02:13:18.259575+00:00
7,run_20260922T021119Z,DATA QUALITY,profil zona silver,OK,3.23,0.19,29.1,2026-09-22T02:13:21.500312+00:00


Spark dihentikan. Proses Java tersisa: 1

Selesai. Lanjut ke 03_mapreduce_yarn.ipynb
